# Final Test Evaluation — ACSIncome (Illinois, 2018)

Evaluate Logistic Regression and Random Forest under four strategies: Baseline, Reweighing, Threshold Optimization, and Exponentiated Gradient. All eight configurations use the same held-out test split. The test labels are not used for fitting models or selecting mitigation rules.

## Data and experimental setup

Reproduce the shared 60/20/20 split. Baseline and Reweighing are trained on Train; Threshold Optimization fits its post-processing rule on Validation; Exponentiated Gradient learns its fairness-constrained classifier mixture on Train.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from fairlearn.metrics import (
    demographic_parity_difference,
    equal_opportunity_difference,
    false_positive_rate,
)
from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.reductions import ExponentiatedGradient, TruePositiveRateParity
from sklearn import set_config
from sklearn.metrics import accuracy_score, balanced_accuracy_score

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import load_acs_income
from src.experiment import prepare_data_splits
from src.models import create_logistic_regression, create_random_forest

In [ ]:
X, y, A = load_acs_income()
X_train, X_val, X_test, y_train, y_val, y_test, A_train, A_val, A_test = (
    prepare_data_splits(X, y, A)
)

model_factories = {
    "Logistic Regression": create_logistic_regression,
    "Random Forest": create_random_forest,
}
test_predictions = {}

## Baseline and Reweighing

Reweighing weights depend only on the joint distribution of income label and `SEX` in Train. Both strategies use the same underlying model pipelines.

In [ ]:
sample_weights = np.ones(len(y_train), dtype=float)

for group in np.unique(A_train):
    group_frequency = np.mean(A_train == group)
    for label in np.unique(y_train):
        mask = (A_train == group) & (y_train == label)
        joint_frequency = np.mean(mask)
        if joint_frequency == 0:
            continue
        label_frequency = np.mean(y_train == label)
        sample_weights[mask] = (
            group_frequency * label_frequency / joint_frequency
        )

baseline_models = {}
for name, factory in model_factories.items():
    baseline = factory()
    baseline.fit(X_train, y_train)
    baseline_models[name] = baseline
    test_predictions[(name, "Baseline")] = np.asarray(baseline.predict(X_test))

    reweighted = factory()
    reweighted.fit(X_train, y_train, classifier__sample_weight=sample_weights)
    test_predictions[(name, "Reweighing")] = np.asarray(reweighted.predict(X_test))

## Threshold Optimization

Fit the group-aware post-processing rule on Validation using true-positive-rate parity and a balanced-accuracy objective. Evaluate the fitted rule only once on Test, with a fixed seed for randomized decisions.

In [ ]:
for name, baseline in baseline_models.items():
    optimizer = ThresholdOptimizer(
        estimator=baseline,
        constraints="true_positive_rate_parity",
        objective="balanced_accuracy_score",
        prefit=True,
        predict_method="predict_proba",
    )
    optimizer.fit(X_val, y_val, sensitive_features=A_val)
    test_predictions[(name, "Threshold Optimization")] = np.asarray(
        optimizer.predict(X_test, sensitive_features=A_test, random_state=42)
    )

## Exponentiated Gradient

Use the same constraint and settings as `05_exponentiated_gradient.ipynb`. The method learns a randomized mixture of classifiers on Train. A fixed seed determines the realized test predictions.

In [ ]:
set_config(enable_metadata_routing=False)

for name, factory in model_factories.items():
    optimizer = ExponentiatedGradient(
        estimator=factory(),
        constraints=TruePositiveRateParity(),
        eps=0.02,
        max_iter=50,
        sample_weight_name="classifier__sample_weight",
    )
    optimizer.fit(X_train, y_train, sensitive_features=A_train)
    test_predictions[(name, "ExponentiatedGradient")] = np.asarray(
        optimizer.predict(X_test, random_state=42)
    )

## Held-out test results

Compute all five metrics from the same test labels and sensitive-group assignments. Fairness gaps are absolute differences, so smaller values indicate smaller observed disparities.

In [ ]:
def evaluate_predictions(name, strategy, predictions):
    labels = np.asarray(y_test)
    groups = np.asarray(A_test)
    male, female = groups == 1, groups == 2

    return {
        "Model": name,
        "Strategy": strategy,
        "Accuracy": accuracy_score(labels, predictions),
        "Balanced Accuracy": balanced_accuracy_score(labels, predictions),
        "EOD": abs(equal_opportunity_difference(
            labels, predictions, sensitive_features=groups
        )),
        "DPD": abs(demographic_parity_difference(
            labels, predictions, sensitive_features=groups
        )),
        "FPR Gap": abs(
            false_positive_rate(labels[male], predictions[male])
            - false_positive_rate(labels[female], predictions[female])
        ),
    }

In [ ]:
strategies = [
    "Baseline", "Reweighing", "Threshold Optimization", "ExponentiatedGradient"
]
final_test_results = pd.DataFrame([
    evaluate_predictions(name, strategy, test_predictions[(name, strategy)])
    for name in model_factories
    for strategy in strategies
])
display(final_test_results.round(4))

results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)
final_test_results.to_csv(results_dir / "final_test_comparison.csv", index=False)

,Model,Strategy,Accuracy,Balanced Accuracy,EOD,DPD,FPR Gap
0,Logistic Regression,Baseline,0.8081,0.7938,0.0342,0.1385,0.0506
1,Logistic Regression,Reweighing,0.8058,0.7911,0.0312,0.0906,0.0128
2,Logistic Regression,Threshold Optimization,0.8034,0.8013,0.0117,0.1274,0.0420
3,Logistic Regression,ExponentiatedGradient,0.8066,0.7923,0.0155,0.1245,0.0398
4,Random Forest,Baseline,0.7931,0.7754,0.0180,0.1221,0.0434
5,Random Forest,Reweighing,0.7940,0.7758,0.0118,0.1019,0.0280
6,Random Forest,Threshold Optimization,0.7856,0.7896,0.0080,0.1054,0.0238
7,Random Forest,ExponentiatedGradient,0.7945,0.7766,0.0279,0.1262,0.0434


## Fairness–accuracy changes relative to Baseline

Report changes separately for each underlying classifier. Accuracy changes are percentage points; EOD reduction is relative to the baseline EOD and can be negative if disparity increases.

In [ ]:
baseline_metrics = final_test_results.loc[
    final_test_results["Strategy"] == "Baseline",
    ["Model", "Accuracy", "EOD"],
]
tradeoffs = final_test_results.merge(
    baseline_metrics, on="Model", suffixes=("", "_Baseline")
)
tradeoffs["Accuracy Change (pp)"] = (
    tradeoffs["Accuracy"] - tradeoffs["Accuracy_Baseline"]
) * 100
tradeoffs["EOD Reduction (%)"] = np.where(
    tradeoffs["EOD_Baseline"] > 0,
    (1 - tradeoffs["EOD"] / tradeoffs["EOD_Baseline"]) * 100,
    np.nan,
)
tradeoff_summary = tradeoffs[
    ["Model", "Strategy", "Accuracy Change (pp)", "EOD Reduction (%)"]
]
display(tradeoff_summary.round(2))
tradeoff_summary.to_csv(results_dir / "tradeoff_summary.csv", index=False)

,Model,Strategy,Accuracy Change (pp),EOD Reduction (%)
0,Logistic Regression,Baseline,0.00,0.00
1,Logistic Regression,Reweighing,-0.23,8.66
2,Logistic Regression,Threshold Optimization,-0.47,65.86
3,Logistic Regression,ExponentiatedGradient,-0.16,54.75
4,Random Forest,Baseline,0.00,0.00
5,Random Forest,Reweighing,0.09,34.24
6,Random Forest,Threshold Optimization,-0.75,55.72
7,Random Forest,ExponentiatedGradient,0.14,-55.40


## Bootstrap confidence intervals

Use 1,000 paired nonparametric bootstrap resamples of the held-out test observations. The same sampled indices are applied to all eight sets of fixed predictions, enabling paired comparisons with Baseline. Report percentile 95% intervals for Accuracy and EOD, and for differences against Baseline.

These intervals describe uncertainty conditional on the fitted models and realized randomized predictions. They do not include variation from retraining or repeated randomized prediction draws.

In [ ]:
def bootstrap_metrics(labels, predictions, groups):
    accuracy = np.mean(labels == predictions)
    tpr = []
    for group in (1, 2):
        positives = (groups == group) & (labels == 1)
        tpr.append(np.mean(predictions[positives]) if positives.any() else np.nan)
    return accuracy, abs(tpr[0] - tpr[1])


rng = np.random.default_rng(42)
labels = np.asarray(y_test)
groups = np.asarray(A_test)
bootstrap_indices = [
    rng.integers(0, len(labels), size=len(labels)) for _ in range(1000)
]

bootstrap_results = {}
for configuration, predictions in test_predictions.items():
    bootstrap_results[configuration] = np.asarray([
        bootstrap_metrics(labels[idx], predictions[idx], groups[idx])
        for idx in bootstrap_indices
    ])

In [ ]:
ci_rows = []
for (name, strategy), values in bootstrap_results.items():
    accuracy_ci = np.nanpercentile(values[:, 0], [2.5, 97.5])
    eod_ci = np.nanpercentile(values[:, 1], [2.5, 97.5])
    ci_rows.append({
        "Configuration": f"{name} | {strategy}",
        "Accuracy CI Lower": accuracy_ci[0],
        "Accuracy CI Upper": accuracy_ci[1],
        "EOD CI Lower": eod_ci[0],
        "EOD CI Upper": eod_ci[1],
    })

bootstrap_ci = pd.DataFrame(ci_rows)
display(bootstrap_ci.round(4))
bootstrap_ci.to_csv(results_dir / "bootstrap_confidence_intervals.csv", index=False)

,Configuration,Accuracy CI Lower,Accuracy CI Upper,EOD CI Lower,EOD CI Upper
0,Logistic Regression | Baseline,0.8016,0.8146,0.0115,0.0593
1,Logistic Regression | Reweighing,0.7991,0.8124,0.0070,0.0566
2,Random Forest | Baseline,0.7860,0.8002,0.0011,0.0428
3,Random Forest | Reweighing,0.7869,0.8011,0.0007,0.0365
4,Logistic Regression | Threshold Optimization,0.7971,0.8098,0.0008,0.0328
5,Random Forest | Threshold Optimization,0.7792,0.7925,0.0004,0.0295
6,Logistic Regression | ExponentiatedGradient,0.7997,0.8127,0.0008,0.0400
7,Random Forest | ExponentiatedGradient,0.7880,0.8022,0.0042,0.0541


In [ ]:
difference_rows = []
for name in model_factories:
    baseline = bootstrap_results[(name, "Baseline")]
    for strategy in strategies[1:]:
        values = bootstrap_results[(name, strategy)]
        accuracy_diff = (values[:, 0] - baseline[:, 0]) * 100
        eod_reduction = baseline[:, 1] - values[:, 1]
        accuracy_ci = np.nanpercentile(accuracy_diff, [2.5, 97.5])
        eod_ci = np.nanpercentile(eod_reduction, [2.5, 97.5])
        difference_rows.append({
            "Model": name,
            "Strategy": strategy,
            "Accuracy Change (pp) Lower": accuracy_ci[0],
            "Accuracy Change (pp) Upper": accuracy_ci[1],
            "EOD Reduction Lower": eod_ci[0],
            "EOD Reduction Upper": eod_ci[1],
        })

difference_ci = pd.DataFrame(difference_rows)
display(difference_ci.round(4))
difference_ci.to_csv(results_dir / "bootstrap_difference_intervals.csv", index=False)

,Model,Strategy,Accuracy Change (pp) Lower,Accuracy Change (pp) Upper,EOD Reduction Lower,EOD Reduction Upper
0,Logistic Regression,Reweighing,-0.5521,0.0970,-0.0451,0.0507
1,Logistic Regression,Threshold Optimization,-0.8430,-0.0597,-0.0000,0.0350
2,Logistic Regression,ExponentiatedGradient,-0.3357,0.0298,0.0052,0.0256
3,Random Forest,Reweighing,-0.2613,0.4625,-0.0316,0.0355
4,Random Forest,Threshold Optimization,-1.2983,-0.2164,-0.0232,0.0364
5,Random Forest,ExponentiatedGradient,-0.1865,0.4702,-0.0220,0.0055


## Interpretation

Compare test performance across strategies and models, considering both fairness gaps and predictive quality. A smaller EOD does not guarantee a smaller DPD or FPR Gap. Because the test set was used only for evaluation, these results provide a more credible comparison than validation scores used for threshold selection. Observed group gaps do not by themselves establish causal discrimination.